## Races Data transform Bronze to Silver

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
# To use reader api options 
races_df = spark.read.table(bronze_table)

# races_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
from pyspark.sql import functions as f

In [0]:
races_selected_df = races_df.select(
    f.col("season"),
    f.col("round"),
    f.col("raceName"),
    f.col("date"),
    f.col("circuitId"),
    f.col("ingestion_timestamp"),
    f.col("source_file")
)

In [0]:
# For column rename we can use ALIAS funcrion or also the below function

column_rename = {
    "circuitId" : "circuit_id",
    "raceName" : "race_name",
    "date" : "race_date"
    }

races_renamed_df = races_selected_df.withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

races_valid_df = races_renamed_df.dropDuplicates(["season", "round"]).filter(((f.col("season").isNotNull()) & (f.col("round").isNotNull())))

In [0]:
races_final_df = races_valid_df.withColumn("race_name", f.initcap("race_name"))

In [0]:
races_final_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)